In [1]:
!unzip -q STANDUP_Database.zip -d /content/

In [2]:
import os, re, cv2, numpy as np, pandas as pd
from pathlib import Path
from tqdm import tqdm
from skimage.feature import graycomatrix, graycoprops

In [3]:
ROOT = "/content/STANDUP_Database"

def parse_index(fname):
    return int(re.findall(r"_(\d+)", fname)[0])

def assign_patient_timepoint(group, idx):
    if group in ["R0", "R1", "R2"]:
        pid = f"{group}_P{(idx-1)//2 + 1}"
        tp = "T0" if (idx-1) % 2 == 0 else "T10"
    else:  # healthy
        if idx <= 39:
            pid, tp = f"H_P{idx}", "single"
        else:
            a = idx - 40
            pid = f"H_P{40 + a//2}"
            tp = "T0" if a % 2 == 0 else "T10"
    return pid, tp

def build_metadata(modality):
    rows = []
    groups = [("diabetic/R0", "R0"), ("diabetic/R1", "R1"), ("diabetic/R2", "R2"), ("healthy", "healthy")]
    for folder, group in groups:
        d = os.path.join(ROOT, folder, modality)
        for fname in sorted(os.listdir(d)):
            idx = parse_index(fname)
            pid, tp = assign_patient_timepoint(group, idx)
            rows.append({"filename": fname, "path": os.path.join(d, fname),
                         "label": group, "patient_id": pid, "timepoint": tp})
    return pd.DataFrame(rows)

rgb_meta = build_metadata("RGB")
thermal_meta = build_metadata("thermal")
print(f"RGB: {len(rgb_meta)} | Thermal: {len(thermal_meta)}")
print(rgb_meta["label"].value_counts())

RGB: 415 | Thermal: 415
label
R0         138
healthy    125
R2          84
R1          68
Name: count, dtype: int64


In [4]:
assert set(rgb_meta["filename"]) == set(thermal_meta["filename"]), "Filename mismatch between RGB and Thermal"
print("All filenames match 1:1 between RGB and Thermal.")

All filenames match 1:1 between RGB and Thermal.


In [5]:
GAMMA, CLIP_LIMIT, TILE_GRID = 1.2, 2.0, (8, 8)

def white_balance(img):
    r = img.copy().astype(np.float32)
    avg_b, avg_g, avg_r = r[:,:,0].mean(), r[:,:,1].mean(), r[:,:,2].mean()
    avg = (avg_b+avg_g+avg_r)/3.0
    r[:,:,0]=np.clip(r[:,:,0]*(avg/avg_b),0,255); r[:,:,1]=np.clip(r[:,:,1]*(avg/avg_g),0,255); r[:,:,2]=np.clip(r[:,:,2]*(avg/avg_r),0,255)
    return r.astype(np.uint8)

def gamma_correction(img, gamma=GAMMA):
    table = np.array([((i/255.0)**(1.0/gamma))*255 for i in range(256)]).astype(np.uint8)
    return cv2.LUT(img, table)

def extract_roi(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array([0,20,70]), np.array([25,255,255]))
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(15,15))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k); mask = cv2.morphologyEx(mask, cv2.MORPH_DILATE, k)
    if mask.sum() < 0.05*mask.size: return img
    return cv2.bitwise_and(img, img, mask=mask)

def apply_clahe(img, clip_limit=CLIP_LIMIT, tile_grid=TILE_GRID):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB); l,a,b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid).apply(l)
    return cv2.cvtColor(cv2.merge([l,a,b]), cv2.COLOR_LAB2BGR)

def preprocess_rgb(img):
    img = white_balance(img); img = gamma_correction(img); img = extract_roi(img); img = apply_clahe(img)
    return cv2.resize(img, (224,224))

In [6]:
OUT_RGB = "/content/STANDUP_RGB_P"
os.makedirs(OUT_RGB, exist_ok=True)
processed, skipped = 0, 0
for _, row in tqdm(rgb_meta.iterrows(), total=len(rgb_meta)):
    img = cv2.imread(row["path"])
    if img is None: skipped += 1; continue
    out = preprocess_rgb(img)
    cv2.imwrite(os.path.join(OUT_RGB, row["filename"]), out)
    processed += 1
print(f"Processed: {processed} | Skipped: {skipped}")

100%|██████████| 415/415 [00:18<00:00, 21.95it/s]

Processed: 415 | Skipped: 0


In [7]:
def noise_reduction(img):
    return cv2.bilateralFilter(img, 9, 75, 75)

def roi_extraction_thermal(img):
    img_u8 = img if img.dtype==np.uint8 else ((img-img.min())/(img.max()-img.min())*255).astype(np.uint8)
    _, thresh = cv2.threshold(img_u8, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    k1 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(15,15)); k2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5))
    closed = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, k1); opened = cv2.morphologyEx(closed, cv2.MORPH_OPEN, k2)
    contours,_ = cv2.findContours(opened, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask = np.zeros_like(opened)
    if contours: cv2.drawContours(mask, [max(contours, key=cv2.contourArea)], -1, 255, -1)
    else: mask = opened
    return cv2.bitwise_and(img_u8, img_u8, mask=mask)

def normalize_thermal(img):
    return (img.astype(np.float32)-img.min())/(img.max()-img.min()+1e-8)

def preprocess_thermal(img):
    img = noise_reduction(img)
    img = roi_extraction_thermal(img)
    img = normalize_thermal(img)
    return cv2.resize(img, (224,224))

In [8]:
OUT_THERM = "/content/STANDUP_Thermal_P"
os.makedirs(OUT_THERM, exist_ok=True)
processed, skipped = 0, 0
for _, row in tqdm(thermal_meta.iterrows(), total=len(thermal_meta)):
    img = cv2.imread(row["path"], cv2.IMREAD_GRAYSCALE)
    if img is None: skipped += 1; continue
    out = preprocess_thermal(img)
    np.save(os.path.join(OUT_THERM, row["filename"].replace(".png",".npy")), out)
    processed += 1
print(f"Processed: {processed} | Skipped: {skipped}")

100%|██████████| 415/415 [00:18<00:00, 22.48it/s]

Processed: 415 | Skipped: 0


In [9]:
def extract_temp_features(img):
    return np.mean(img), np.max(img), np.std(img), np.max(img)-np.min(img)

def extract_spatial_features(img):
    thresh = np.percentile(img, 90)
    hot = (img > thresh).astype(np.uint8)
    n_labels,_ = cv2.connectedComponents(hot)
    img_u8 = (img*255).astype(np.uint8)
    gx = cv2.Sobel(img_u8, cv2.CV_64F,1,0,ksize=3); gy = cv2.Sobel(img_u8, cv2.CV_64F,0,1,ksize=3)
    grad = np.mean(np.sqrt(gx**2+gy**2))
    return n_labels-1, grad

def extract_texture_features(img):
    img_u8 = (img*255).astype(np.uint8)
    glcm = graycomatrix(img_u8, [1], [0,np.pi/4,np.pi/2,3*np.pi/4], levels=256, symmetric=True, normed=True)
    contrast = graycoprops(glcm,'contrast')[0,0]; homogeneity = graycoprops(glcm,'homogeneity')[0,0]
    g = glcm/(glcm.sum()+1e-10); g = g[g>0]
    entropy = -np.sum(g*np.log2(g))
    return contrast, homogeneity, entropy

In [10]:
rows = []
for _, row in tqdm(thermal_meta.iterrows(), total=len(thermal_meta)):
    npy_path = os.path.join(OUT_THERM, row["filename"].replace(".png",".npy"))
    img = np.load(npy_path)
    mean_t, max_t, std_t, range_t = extract_temp_features(img)
    hot_spot_count, temp_gradient = extract_spatial_features(img)
    glcm_contrast, glcm_homogeneity, glcm_entropy = extract_texture_features(img)
    rows.append({**row.to_dict(), "mean_temp": mean_t, "max_temp": max_t, "std_temp": std_t,
                 "range_temp": range_t, "hot_spot_count": hot_spot_count, "temp_gradient": temp_gradient,
                 "glcm_contrast": glcm_contrast, "glcm_homogeneity": glcm_homogeneity, "glcm_entropy": glcm_entropy})

standup_thermal_features = pd.DataFrame(rows)
standup_thermal_features.to_csv("/content/standup_thermal_features.csv", index=False)
rgb_meta.to_csv("/content/standup_rgb_metadata.csv", index=False)
print(standup_thermal_features.shape)
print(standup_thermal_features.head())

100%|██████████| 415/415 [00:12<00:00, 32.85it/s]

(415, 14)
      filename                                               path label  \
0  R0_0001.png  /content/STANDUP_Database/diabetic/R0/thermal/...    R0   
1  R0_0002.png  /content/STANDUP_Database/diabetic/R0/thermal/...    R0   
2  R0_0003.png  /content/STANDUP_Database/diabetic/R0/thermal/...    R0   
3  R0_0004.png  /content/STANDUP_Database/diabetic/R0/thermal/...    R0   
4  R0_0005.png  /content/STANDUP_Database/diabetic/R0/thermal/...    R0   

  patient_id timepoint  mean_temp  max_temp  std_temp  range_temp  \
0      R0_P1        T0   0.067864  0.996780  0.209197    0.996780   
1      R0_P1       T10   0.096387  0.996115  0.205166    0.996115   
2      R0_P2        T0   0.124021  0.998133  0.251631    0.998133   
3      R0_P2       T10   0.033476  0.999831  0.143026    0.999831   
4      R0_P3        T0   0.030296  1.000000  0.140658    1.000000   

   hot_spot_count  temp_gradient  glcm_contrast  glcm_homogeneity  \
0               1      10.454853      47.839246        